# 01 — Check the environment and introduce Qwen

This notebook verifies the terminal/Jupyter environment, checks CUDA, safely imports NeMo when installed, and introduces the two model paths used by the series. It does **not** install packages or download weights automatically.

- Primary NeMo recipe: `Qwen/Qwen3-1.7B`
- Smaller optional local inference model: `Qwen/Qwen2.5-1.5B-Instruct`

The checks are laptop-safe. The optional inference cell is disabled because it downloads model files and can use several GB of RAM and disk.

In [1]:
from pathlib import Path
import platform
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

print('Project root:', PROJECT_ROOT)
print('Platform:', platform.platform())
subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts' / 'check_gpu.py')], check=True)

Project root: C:\Code\finetuning\nvftqwen\nemo-qwen-hands-on-series
Platform: Windows-11-10.0.26200-SP0


CompletedProcess(args=['C:\\Code\\finetuning\\nvftqwen\\nemo-qwen-hands-on-series\\.venv\\Scripts\\python.exe', 'C:\\Code\\finetuning\\nvftqwen\\nemo-qwen-hands-on-series\\scripts\\check_gpu.py'], returncode=0)

## Installation choices

Run installation commands in a terminal, then restart this notebook's kernel.

```bash
# Laptop-safe notebook/data environment
python -m pip install -r requirements-local.txt

# NeMo-oriented environment on a compatible Linux/CUDA host
python -m pip install -r requirements.txt
```

For a real NeMo run, NVIDIA's NeMo container is often more reproducible than mixing arbitrary host CUDA and Python packages. Check the current NeMo installation documentation before choosing an image or PyTorch build.

In [2]:
from importlib import metadata

for package in ['torch', 'transformers', 'accelerate', 'nemo-toolkit', 'nemo-run', 'jupyterlab']:
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        version = 'not installed'
    print(f'{package:18} {version}')

try:
    import nemo
    print('NeMo import: OK', getattr(nemo, '__version__', '(version unavailable)'))
except Exception as exc:
    print('NeMo import: unavailable in this kernel')
    print('Detail:', type(exc).__name__, exc)

torch              2.13.0
transformers       5.13.0
accelerate         1.14.0
nemo-toolkit       not installed
nemo-run           not installed
jupyterlab         4.6.1
NeMo import: unavailable in this kernel
Detail: ModuleNotFoundError No module named 'nemo'


## Why two Qwen models?

NeMo 2.0 provides recipe families for Qwen3, including the 1.7B dense model. That is the current training path used in Notebook 3. Qwen2.5-1.5B-Instruct is retained as a small, widely supported Hugging Face option for learning the local generation API.

A model identifier is only a pointer. Downloading weights requires network access and storage; running them requires enough host or GPU memory. The next cell only prints identifiers unless its flag is changed.

In [3]:
NEMO_MODEL_ID = 'Qwen/Qwen3-1.7B'
LOCAL_MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
CHECK_REMOTE_CONFIG = False  # Opt in: performs a small Hugging Face network request.

print('NeMo path:', NEMO_MODEL_ID)
print('Local option:', LOCAL_MODEL_ID)

if CHECK_REMOTE_CONFIG:
    from transformers import AutoConfig
    config = AutoConfig.from_pretrained(NEMO_MODEL_ID, trust_remote_code=False)
    print(config)
else:
    print('Remote config check skipped. Set CHECK_REMOTE_CONFIG=True to enable it.')

NeMo path: Qwen/Qwen3-1.7B
Local option: Qwen/Qwen2.5-1.5B-Instruct
Remote config check skipped. Set CHECK_REMOTE_CONFIG=True to enable it.


In [4]:
RUN_LOCAL_INFERENCE = True  # Downloads weights; review RAM/VRAM and disk first.

if RUN_LOCAL_INFERENCE:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_ID, trust_remote_code=False)
    model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_ID,
        torch_dtype='auto',
        device_map='auto',
        trust_remote_code=False,
    )
    messages = [{'role': 'user', 'content': 'Explain LoRA in two short sentences.'}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=96, do_sample=False)
    new_tokens = generated[0, inputs.input_ids.shape[1]:]
    print(tokenizer.decode(new_tokens, skip_special_tokens=True))
else:
    print('Local inference skipped. This is the safe default.')

C:\Code\finetuning\nvftqwen\nemo-qwen-hands-on-series\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Code\finetuning\nvftqwen\nemo-qwen-hands-on-series\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bbalakreshna\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or 

LoRA stands for Low-Rank Adaptation and is a technique used to improve the performance of deep learning models by adding low-rank matrices to their weights. It allows for efficient parameter sharing between layers while still enabling fine-tuning on new tasks or datasets.


## Next

Continue to Notebook 2 to validate both supported JSONL schemas and normalize them to a common in-memory chat representation.